# MLP hyperparameter search (pl19_800 only)

Part C of `workbench.ipynb` for the reference 19-port lantern at 800 nm, split
out so that the workbench itself runs for every lantern preset. It reads the
results of `scripts/mlp_hpo.py` (`data/ml/mlp_hpo/`), which still names its
scenarios and datasets after this lantern; porting it to other lanterns means
adapting that script and running it (a few hours on one GPU).


In [ ]:
import os
os.environ["CBEAM_BACKEND"] = "numpy"

import numpy as np
import matplotlib.pyplot as plt
import ml_reconstruct as ml
from ml_reconstruct.notebook import *            # helpers: datasets, scoring, plots, fit-or-load

# the validation pistons of the workbench's Part A and Part B datasets
CA_va = ml.LanternDataset.load("data/ml/single_800nm.npz").val[1]
CB_va = ml.LanternDataset.load("data/ml/spectral_41wl.npz").val[1]


# Part C: what the MLP alone can achieve

MLPs at every sampling of the wavelength axis:

| scenario | inputs | pistons | |
|---|---|---|---|
| `800nm` | 19 cores at 800 nm | +-400 nm (+-lambda/2) | Part A's dataset: inside the single-wavelength capture range |
| `1wl-800` | 19 cores at 800 nm | +-6.4 um (+-8 lambda) | one wavelength of Part B's dataset: far outside it |
| `20nm` | 19 cores x 11 wavelengths (700-900 nm, 20 nm steps) | +-6.4 um (+-8 lambda) | |
| `10nm` | 19 x 21 wavelengths (10 nm steps) | +-6.4 um (+-8 lambda) | |
| `5nm` | 19 x 41 wavelengths (5 nm steps) | +-6.4 um (+-8 lambda) | Part B's full spectrum |

With several wavelengths there are two approaches:

- **Full-spectrum MLP.** One network on the channel powers of all the
  scenario's wavelengths, regressing the pistons directly (as in B2).
- **Per-wavelength MLPs + unwrap.** One small network per wavelength on its 19
  channel powers, regressing each piston's phasor (sin, cos)(2 pi p / lambda),
  that is the piston wrapped at that wavelength. The multi-wavelength unwrap
  of B4 then fuses the wavelengths. It is B3/B4 with MLPs in place of the OMP
  polynomials. The 20 and 10 nm wavelengths are subsets of the 41, so one set
  of 41 per-wavelength models serves all three spectral scenarios.

`scripts/mlp_hpo.py` runs a small hyperparameter search for both, to balance
size against accuracy:

- **Full-spectrum grid:** depth {2, 3, 4} x width {32, 64, 128, 256}, Adam
  with a learning rate decaying from 3e-4 on a cosine schedule. That's 2000
  epochs at 800 nm and 5000 with the spectra. A first attempt starting from
  1e-3 generalised much worse on the spectra (17-39 nm), so it was dropped;
  its two runs are in `data/ml/mlp_hpo_lr1e-3/`.
- **Reference runs:** the notebooks' recipe (constant 3e-4) on their
  architectures, and one 4 x 256 network in `1wl-800`.
- **Seed stage:** the best and the "balanced" full-spectrum configuration of
  each scenario (the fewest parameters within 10% of the best RMSE),
  retrained with two more seeds.
- **Per-wavelength MLPs:** 2 x 32, 2 x 64, 3 x 64 and 3 x 128, 2000 epochs at
  each of the 41 wavelengths, then unwrapped in each spectral scenario.

The best epochs, the picked configurations and the per-wavelength error
estimates the unwrap weights by all come from the last 10% of the training
split, held out from training. The validation split is used only to report,
so these numbers are not tuned on what they are scored on. The models of
Parts A and B did pick their best epoch on the validation split, and they
trained on 10% more rows. Run the search (a few hours on one GPU) with

    setsid nohup bash scripts/run_mlp_hpo.sh > data/ml/mlp_hpo/run.log 2>&1 < /dev/null & disown

This part only reads its results (`data/ml/mlp_hpo/`) and shows whatever has
finished.

In [ ]:
HPO_DIR = "data/ml/mlp_hpo"
SCEN_C = {"800nm": "1 wl (800 nm), +-400 nm", "1wl-800": "1 wl (800 nm), +-6.4 um",
          "20nm": "11 wl, 20 nm steps", "10nm": "21 wl, 10 nm steps", "5nm": "41 wl, 5 nm steps"}
C_VAL_C = {"800nm": CA_va, "1wl-800": CB_va, "20nm": CB_va, "10nm": CB_va, "5nm": CB_va}
runs = load_hpo_runs(HPO_DIR)
print(f"{len(runs)} finished runs:", dict(runs.groupby("role").size()) if len(runs) else "none")

## C1. Full-spectrum MLP: the grid

Validation RMSE (seed 0) and parameter count of every grid configuration;
rows are depth x width.

In [ ]:
display(hpo_pivot(runs, "val_rmse_mean", SCEN_C).round(1), hpo_pivot(runs, "n_params", SCEN_C))

## C2. Per-wavelength MLPs

The wrapped-piston error of each architecture at each wavelength (to compare
with the OMP models of B3), then the unwrapped result per spectral scenario.
The parameters are summed over the scenario's wavelengths.

In [ ]:
plot_perwl_wrapped(runs)
display(hpo_unwrap_table(runs))

## C3. Accuracy vs. size, both approaches

One panel per scenario. Each coloured line is one depth of the full-spectrum
grid, labelled with its widths. The star is the notebook recipe, and the
picked configurations are circled (red: best, green: balanced). The black
diamonds are the per-wavelength + unwrap architectures.

In [ ]:
scen_grid = [s for s in ("800nm", "20nm", "10nm", "5nm") if s in set(runs[runs.role == "grid"].scenario)]
plot_hpo_grid(runs, scen_grid, [SCEN_C[s] for s in scen_grid])

## C4. The best of each approach, per scenario

The full-spectrum picks averaged over three seeds (the error bars are the
seed-to-seed range of the mean RMSE), the reference runs, the
single-wavelength run over +-5 um, and the per-wavelength + unwrap
architecture with the lowest held-out RMSE. "gross %" is the share of
estimates off by more than 350 nm: half the shortest wavelength, so a wrong
fringe in the spectral scenarios.

In [ ]:
summary_C = hpo_seed_summary(runs, C_VAL_C, HPO_DIR)
summary_C["scenario"] = summary_C.scenario.astype("category").cat.set_categories(list(SCEN_C))
summary_C = summary_C.sort_values(["scenario", "pick"]).reset_index(drop=True)
display(summary_C.round({"val_rmse": 2, "seed_min": 2, "seed_max": 2, "train_s": 0, "gross_%": 3}))
plot_hpo_summary(summary_C, SCEN_C)

## C5. Estimates of the best models (seed 0)

All 5 pistons of every validation sample, estimate against truth.

In [ ]:
plot_hpo_estimates(summary_C[summary_C.pick.isin(["best", "single-wl"])], runs, C_VAL_C, SCEN_C, HPO_DIR)
plot_hpo_estimates(summary_C[summary_C.pick == "per-wl + unwrap"], runs, C_VAL_C, SCEN_C, HPO_DIR)

## C6. One wavelength: more data vs. bigger models

At 800 nm the search's best configuration (4 x 256, cosine from 3e-4, 2000
epochs) roughly halves the error of the notebook's MLP. Beyond it, quick
one-off tests on the 10000-sample dataset (seed 0, same protocol; not rerun
here) changed little:

| change | validation RMSE |
|---|---|
| 4 x 256, 2000 epochs (the search's best) | 4.27 nm |
| 6000 epochs | 4.04 nm |
| 5 x 256 | 3.98 nm |
| 4 x 512 | 4.14 nm |
| 5 x 256, 6000 epochs | 4.32 nm |
| batch size 64 instead of 128 | 4.30 nm |
| log or square root of the signals | 11.1 / 6.3 nm |
| ensemble of 5 seeds (mean of their estimates) | 3.79 nm |

The held-out loss stays about twice the training loss, so the model is
limited by its 7200 training samples, not by its size. The 800 nm dataset is
cheap to simulate (32 minutes for 50000 samples on the CPU), so
`data/gen/gen_single_50k.py` made one of 50000 samples
(`data/ml/single_800nm_50k.npz`, the same `rng=0` draw, so its first 10000
coefficients are those of Part A). The search's `data800` stage trains the
same 4 x 256 network with 5 seeds on its first 7200, 18000 and 36000 rows,
selecting on its held-out rows 36000-40000 and scoring on its 10000
validation rows.

On those 10000 validation rows the errors turn out to be heavy-tailed. The
median error is a fraction of a nm, but a few configurations in a thousand
are off by tens to hundreds of nm, and they dominate the RMSE. That also
makes any RMSE on 2000 validation rows (Part A, and the `800nm` scenario
above) noisy: the 4 x 256 network trained on the same 7200 rows scores
4.3 nm on Part A's 2000 rows and 9.9 nm on these 10000. So the table also
gives the quantiles of |error| over all pistons. Every model trained on the
first 7200 rows (the notebook recipe included) is rescored here on the same
10000 rows.

In [ ]:
import pandas as pd
D50K = "data/ml/single_800nm_50k.npz"
ds50 = ml.LanternDataset.load(D50K)
VA50, CV50 = slice(ds50.n_train, None), ds50.val[1]
seeds_800 = lambda n: [f"800nm-50k-d4w256-cos3e-4-e2000-n{n}-s{s}" for s in range(5)]
models_800 = {"notebook recipe (3x128, 500 ep), 7200": (["800nm-d3w128-const3e-4-e500-s0"], 7200),
              "4x256 cosine 2000 ep, 7200": (seeds_800(7200)[:1], 7200),
              "4x256, 7200, ensemble of 5": (seeds_800(7200), 7200),
              "4x256, 18000, ensemble of 5": (seeds_800(18000), 18000),
              "4x256, 36000": (seeds_800(36000)[:1], 36000),
              "4x256, 36000, ensemble of 5": (seeds_800(36000), 36000)}

In [ ]:
robust_800 = pd.DataFrame({k: error_stats(rescore_mlp(ids, HPO_DIR, ds50, VA50, n), CV50)
                           for k, (ids, n) in models_800.items()}).T
display(robust_800.round(2))

In [ ]:
table_800 = data800_table(runs, CV50, HPO_DIR)
display(table_800.round(2))
plot_data800(table_800, ref=[(7200, robust_800.rmse.iloc[0], "notebook recipe")])

## C7. Findings

**One wavelength (800 nm, +-400 nm).** The search's best network, 4 x 256
with a cosine schedule over 2000 epochs, halves the notebook MLP's error. It
scores 4.4 nm over three seeds (4.3-4.6) against 9.7 nm, on Part A's 2000
validation rows. On the 10000-row validation set the gain is larger than
the RMSE shows:

- **Median error:** 3.4 nm becomes 1.0 nm.
- **99th percentile:** 41 nm becomes 18 nm.

With 5x the training data and an ensemble of 5 seeds, it reaches a median of
0.27 nm and a 99th percentile of 2.7 nm, with only 0.06% of the estimates
off by more than 50 nm. The RMSE (13.6 -> 6.7 nm) is held up by those rare
outliers. Bigger networks, longer training and other input transforms did
not help; more data did.

**One wavelength over +-6.4 um.** No skill (2910 nm, against 2893 nm for
predicting the mean): a single wavelength sees a piston only modulo lambda.

**Several wavelengths, full-spectrum MLP.** It works, but it is hard to
train:

| sampling | best RMSE | seed range |
|---|---|---|
| 20 nm | 63 nm | |
| 10 nm | 17.5 nm | 10-22 nm |
| 5 nm | 14.4 nm | 8-19 nm |

The networks need depth (4 layers): 3-layer ones stay at about 50 nm at 5 nm
steps. The best epoch comes before the end of the cosine schedule, and the
notebook's constant-rate 10000-epoch recipe (13.8 nm) is no worse. Accuracy
here is limited by optimisation and seed variance, not by size.

**Several wavelengths, per-wavelength MLPs + unwrap.** Far better, and
stable:

| sampling | 3 x 128 per wavelength | 3 x 64 per wavelength |
|---|---|---|
| 20 nm | 0.82 nm | 1.8 nm (113k parameters) |
| 10 nm | 0.61 nm | 1.4 nm |
| 5 nm | 0.44 nm | 1.0 nm |

There are no wrong fringes in any of them, against 1.64 nm for the OMP
version of B4. Each per-wavelength model only has to learn a smooth,
bounded function (the phasor) over one fringe, and the unwrap does the rest
by construction. Even 20 nm steps with 3 x 64 networks (113k parameters in
all) beat every full-spectrum MLP by an order of magnitude, including the
5 nm 4 x 256 one (398k). The error still falls with the network size
(wrapped RMSE per wavelength 23 -> 11 -> 6.7 -> 3.0 nm), so larger
per-wavelength networks are the obvious next step.

**Piston 2** is the easiest piston everywhere except in the full-spectrum
MLPs: about 3 nm against 8 nm at 800 nm with 36000 samples, and the lowest
wrapped error at every wavelength. `piston2.ipynb` looks into why.